In [135]:
from pathlib import Path
from io import StringIO
import pandas as pd
import re

In [136]:
DIR = Path("Consolidated_Schedule_of_Investments/2026-03-31/")
DIR_FILES = Path(DIR, "soi_forms")
OUTDIR = Path(DIR, "outputs")
OUTDIR.mkdir(exist_ok=True)

In [137]:
SCHEDULE_ANCHORS = [
    'id="consolidated_schedule_investments"',
    "Consolidated Schedule of Investments"
]

END_MARKERS = [
    "Consolidated Statement of Assets and Liabilities",
    '<div style="z-index:2;flex-direction:column;display:flex;padding-bottom:0.3in;min-height:0.5in;justify-content:flex-end;position:relative;box-sizing:border-box;"><p style="font-size:10pt;margin-top:12pt;font-family:Times New Roman;margin-bottom:0;text-align:center;margin-left:0;margin-right:0;"><span style="color:#000000;white-space:pre-wrap;font-size:10pt;font-family:Times New Roman;font-kerning:none;min-width:fit-content;">F-</span><span style="color:#000000;white-space:pre-wrap;font-size:10pt;font-family:Times New Roman;font-kerning:none;min-width:fit-content;">26</span></p></div>',
    "Statement of Assets and Liabilities",
    "Consolidated Statement of Operations",
    "Commitment Type",
    "Statement of Operations",
    # 'ix:footnote id="fn-1"',
    # "Financial Highlights",
]

industry_list = (
    "Aerospace and Defense",
    "Auto Components",
    "Automobiles",
    "Banks",
    "Beverages",
    "Biotechnology",
    "Building Products",
    "Capital Markets",
    "Chemicals",
    "Commercial Services and Supplies",
    "Communications Equipment",
    "Construction and Engineering",
    "Construction Materials",
    "Consumer Finance",
    "Consumer Staples Distribution and Retail",
    "Containers and Packaging",
    "Distributors",
    "Diversified Consumer Services",
    "Diversified Financial Services",
    "Diversified Telecommunication Services",
    "Electric Utilities",
    "Electrical Equipment",
    "Electronic Equipment, Instruments and Components",
    "Energy Equipment and Services",
    "Entertainment",
    "Financial Services",
    "Food Products",
    "Gas Utilities",
    "Ground Transportation",
    "Health Care Equipment and Supplies",
    "Health Care Technology",
    "Hotels, Restaurants and Leisure",
    "Household Durables",
    "Healthcare Providers and Services", 
    "Household Products",
    "Industrial Conglomerates",
    "Insurance",
    "Interactive Media and Services",
    "Internet and Catalog Retail",
    "Internet and Direct Marketing Retail",
    "Internet Software and Services",
    "IT Services",
    "Life Sciences Tools and Services",
    "Machinery",
    "Media",
    "Metals and Mining",
    "Oil, Gas and Consumable Fuels",
    "Paper and Forest Products",
    "Pharmaceuticals",
    "Professional Services",
    "Real Estate Management and Development",
    "Road and Rail",
    "Semiconductors and Semiconductor Equipment",
    "Software",
    "Specialty Retail",
    "Trading Companies and Distributors",
    "Textiles, Apparel and Luxury Goods",
    "Technology Hardware, Storage and Peripherals",
    "Transportation Infrastructure",
    "Wireless Telecommunication Services",
)


instrument = (
    "First Lien Debt",
    "Second Lien Debt",
    "Unsecured Debt",
    "Structured Finance Obligations",
    "Equity and other"
)


investment_type = (
    "Investments at Fair Value",
    "Short-Term Investments",
    "Debt Investments(A)",
    "Equity Securities",
    "First Lien Debt - non-controlled/non-affiliated",
    "First Lien Debt - non-controlled/affiliated",
    "First Lien Debt - controlled/affiliated",
    "Second Lien Debt - non-controlled/non-affiliated",
    "Unsecured Debt - non-controlled/non-affiliated",
    "Structured Finance Obligations - Debt Instruments - non-controlled/non-affiliated",
    "Structured Finance Obligations - Equity Instruments - non-controlled/non-affiliated",
    "Equity and other - non-controlled/non-affiliated",
    "Equity and other - non-controlled/affiliated",
    "Equity and other - controlled/affiliated (excluding Investments in Joint Ventures)",
    "Investments in Joint Ventures",
    "Cash and Cash Equivalents",
)

In [138]:
def extract_schedule_html(text: str) -> str:
    start = -1
    
    # returns the index of the first match if found, otherwise it returns -1
    for anchor in SCHEDULE_ANCHORS:
        idx = text.find(anchor)
        if idx != -1:
            # print(f"Start: {anchor} --> {idx}")
            start = idx
            break          
    if start == -1:
        return ""

    # finding ending point
    end_candidates = [] # multiple end points
    for marker in END_MARKERS:
        idx = text.find(marker, start) # Search after start position.
        if idx != -1:
            # print(f"End: {marker} --> {idx}")
            end_candidates.append(idx)
            
    # if markers found choose earliest one otherwise file length
    end = min(end_candidates) if end_candidates else len(text)
    # return the portion of that file
    return text[start:end]

In [139]:
def combine_currency_amount(vals, start_index, span: int = 3) -> str:
    if start_index is None:
        return ""

    parts = []
    for idx in range(start_index, min(start_index + span, len(vals))):
        part = clean(vals[idx])
        if not part or part == "$":
            continue
        parts.append(part)

    if not parts:
        return ""

    combined = "".join(parts)
    combined = combined.replace("$", "").replace().strip()
    combined = re.sub(r"\)([^)\d].*)$", ")", combined)

    if combined.startswith("(") and not combined.endswith(")"):
        combined = f"{combined})"

    return combined.strip()

In [145]:
def clean(x):
    if pd.isna(x):
        return ""

    text = (
        str(x)
        .replace("\xa0", " ")
        .replace("\u200b", "")
        .replace("�", "")
    )

    return re.sub(r"\s+", " ", text).strip()

def normalize_header_label(label: str) -> str:
    val = re.sub(r"\s+", " ", label).strip().upper()
    return val


def filing_part_name(filepath: Path) -> str:
    return filepath.stem 


def pick_value(vals, index):
    if index is None or index >= len(vals):
        return ""
    return vals[index]

def is_blank_row(vals) -> bool:
    return all(v == "" for v in vals)


# from itertools import groupby

def is_section_row(vals, positions) -> bool:
    portfolio_company = pick_value(vals, positions["portfolio_company"])
    if not portfolio_company:
        return False

    values = [v for v in vals if v]
    if not values:
        return False
    
    return all(v == values[0] or v in ["$", "—", "�"] for v in values)


def normalize_section(text):
    return clean(text).replace("(continued)", "").strip().lower()

    
def classify_section(text: str) -> str | None:
    norm = normalize_section(text)
    
    if any(normalize_section(x) == norm for x in industry_list):
        return "industry", norm
    
    elif any(normalize_section(x) == norm for x in instrument):
        return "instrument", norm
    
    elif any(normalize_section(x) == norm for x in investment_type):
        return "investment type", norm
        
    return None, text


def combine_fair_value(vals, start_index, footnotes_index) -> str:
    amount = combine_currency_amount(vals, start_index, span=2)
    if amount.startswith("(") and not amount.endswith(")"):
        close_cell = pick_value(vals, footnotes_index)
        if close_cell.startswith(")"):
            amount = f"{amount})"
    return amount


def infer_investment_type(asset_class: str, industry: str, investment_type: str) -> str:
    if investment_type:
        return investment_type

    sections = f"{asset_class} {industry}"
    for inferred, markers in INVESTMENT_TYPE_BY_SECTION:
        if any(marker in sections for marker in markers):
            return inferred
    return "Other"


def parse_footnotes(vals, footnotes_index) -> str:
    footnotes = pick_value(vals, footnotes_index)
    footnotes = FOOTNOTE_SUFFIX_PAT.sub("", footnotes).lstrip(")").strip()
    return footnotes


def is_skip_text(text: str) -> bool:
    t = text.lower()
    return (
        not t
        or "accompanying notes" in t
        or "consolidated schedule of investments" in t
        or t.startswith("total ")
        or t == "portfolio company"
    )


def normalize_numeric(v: str):
    v = clean(v)
    if v.startswith("(") or v.endswith(")"):
        v = f"-{v.strip('()')}"

    if v in {"", "-", "—"}:
        return pd.NA
   
    return v


def combine_rate_spread(v:str, v2:str):
    return v + " " + v2

In [141]:
def detect_header_positions(df: pd.DataFrame):
    for ridx in range(min(5, len(df))):
        vals = [clean(v) for v in df.iloc[ridx].tolist()]
        normalized = [normalize_header_label(v) for v in vals if v]
        
        has_portfolio = any(v.startswith("PORTFOLIO COMPANY") for v in normalized)
        has_industry= any(v.startswith("INDUSTRY") for v in normalized)
        has_security = any("SECURITY(1)" in v for v in normalized)
        has_interset = any(v.startswith("INTEREST RATE(2)") for v in normalized)
        has_acquisition_date = any(v.startswith("INITIAL ACQUISITION DATE") for v in normalized)
        has_maturity = any(v.startswith("MATURITY DATE") for v in normalized)
        has_amount_quantity = any(v.startswith("PAR AMOUNT / QUANTITY") for v in normalized)
        has_cost = any(v.startswith("COST") for v in normalized)
        has_fair_value = any(v.startswith("FAIR VALUE") for v in normalized)
        has_class_percentage = any(v.startswith("PERCENTAGE OF CLASS(3)") for v in normalized)

        if has_portfolio and has_industry and has_security and has_interset and has_acquisition_date and has_maturity and has_cost and has_fair_value:
            label_map = {normalize_header_label(v): i for i, v in enumerate(vals) if v}
            
            def find_index(*candidates: str):
                for candidate in candidates:
                    idx = label_map.get(candidate)
                    if idx is not None:
                        return idx
                return None

            def leftmost_label_index(*labels: str):
                idxs = [
                    i
                    for i, v in enumerate(vals)
                    if normalize_header_label(v) in labels
                    or any(normalize_header_label(v).startswith(label) for label in labels)
                ]
                return min(idxs) if idxs else None


            obj = { 
                "header_row": ridx, 
                "portfolio_company": find_index("PORTFOLIO COMPANY"), 
                "industry": find_index("INDUSTRY"),
                "security": find_index("SECURITY(1)"),
                "interest_rate": find_index("INTEREST RATE(2)"),
                "acquisition_date": find_index("INITIAL ACQUISITION DATE"),
                "amount_quantity": find_index("PAR AMOUNT / QUANTITY"), 
                "maturity": find_index("MATURITY DATE"), 
                "percentage_of_class": find_index("PERCENTAGE OF CLASS(3)"),
                "cost": find_index("COST"), 
                "fair_value": find_index("FAIR VALUE"),
            }

            
            return obj
    return None

In [143]:
def parse_table(
    df: pd.DataFrame,
    part: str,
    table_index: int,
    investment_type: str = ""
    # industry: str = "", 
    # instrument: str = ""
):
    positions = detect_header_positions(df)
    if not positions:
        return [], [], asset_class, industry
        
    rows = []
    rejects = []
    # print("positions", positions)
    for row_index in range(positions["header_row"], len(df)):
        vals = [clean(v) for v in df.iloc[row_index].tolist()]
        if is_blank_row(vals):
            continue

        portfolio_company = pick_value(vals, positions["portfolio_company"])
        if is_section_row(vals, positions):
            section_kind, sec = classify_section(portfolio_company)
            if section_kind == "investment type":
                investment_type = sec
            # elif section_kind == "instrument":
            #     instrument = sec
            # elif section_kind == "industry":
            #     industry = sec
            continue


        joined = " | ".join([v for v in vals if v])
        upper = joined.upper()
        
        if "ISSUER" in upper and "FAIR VALUE" in upper:
            continue
        
        portfolio_company = pick_value(vals, positions.get("portfolio_company"))
        industry = pick_value(vals, positions.get("industry"))
        security = pick_value(vals, positions.get("security"))
        interest_rate = pick_value(vals, positions.get("interest_rate"))
        acquisition_date = pick_value(vals, positions.get("acquisition_date"))
        percentage_of_class = pick_value(vals, positions.get("percentage_of_class"))
        maturity = pick_value(vals, positions.get("maturity"))
        amount_quantity = pick_value(vals, positions.get("amount_quantity"))
        cost = pick_value(vals, positions.get("cost"))
        fair_value = pick_value(vals, positions.get("fair_value"))
  
        
        cost_ok = bool(NUM_PAT.match(cost)) or cost in {"", "-", "—"}
        fair_ok = bool(NUM_PAT.match(fair_value)) or fair_value in {"", "-", "—"}

        
        if (
            portfolio_company
            and not is_skip_text(portfolio_company)
            and cost_ok
            and fair_ok
        ):
            
            obj = { 
                "portfolio_company": portfolio_company, 
                "industry": industry, 
                "security": security, 
                "interest_rate": interest_rate, 
                "investment_type": investment_type, 
                "acquisition_date": acquisition_date, 
                "percentage_of_class": percentage_of_class, 
                "maturity": maturity, 
                "amount_quantity": normalize_numeric(amount_quantity), 
                "cost": normalize_numeric(cost),
                "fair_value": normalize_numeric(fair_value),
                "part": part, 
                "table_index": table_index, 
                "row_index": row_index 
            }
            print("Normalize Cost: ",normalize_numeric(cost))
            if pd.isna(obj["cost"]) or obj["cost"] == "":
                obj["cost"] = 0
                
            if pd.isna(obj["fair_value"]) or obj["fair_value"] == "":
                obj["fair_value"] = 0
                    
            rows.append(obj)
            
        else:
            if joined and not is_skip_text(portfolio_company):
                rejects.append({
                    "issuer": portfolio_company,
                    "part": part,
                    "table_index": table_index,
                    "row_index": row_index,
                    "raw_row": joined,
                })
    
    target_portfolio_company = {"Other Cash and Cash Equivalents", "Cash and Cash Equivalents - 16.7% of Net Assets"}
    rows = [o for o in rows if o.get("portfolio_company") not in target_portfolio_company]

    return rows, rejects, investment_type

In [144]:
def main():
    all_clean = []

    for filepath in sorted(DIR_FILES.iterdir()):
        if filepath.suffix.lower() not in {".htm", ".html"}:
            continue
        filename = str(filepath).split("/")[-1].split(".")[0]
        part = filing_part_name(filepath)
        html = filepath.read_text(errors="ignore")
        snippet = extract_schedule_html(html)
        tables = pd.read_html(StringIO(snippet), flavor="lxml")
        
        part_rows = []
        candidate_tables = 0
        investment_type = industry = instrument = ""
        
        # print(tables[1])
        
        for table_index, df in enumerate(tables):
            # if table_index == 1: break
            if detect_header_positions(df):
                candidate_tables += 1
                rows, rejects, investment_type = parse_table(
                    df, part, table_index, investment_type
                )
                part_rows.extend(rows)
                
            elif detect_piv_header_positions(df):
                candidate_tables += 1
                rows, rejects, asset_class, industry = parse_piv_table(
                    df, part, table_index, asset_class, industry
                )
                part_rows.extend(rows)

        part_df = pd.DataFrame(part_rows)
        if not part_df.empty:
            part_df = part_df.sort_values(["table_index", "row_index"], kind="stable").reset_index(drop=True)

        all_clean.append(part_df)
        master_df = pd.concat(all_clean, ignore_index=True) if all_clean else pd.DataFrame()

        dedup_cols = [
            "portfolio_company",
            "industry",
            "security",
            "interest_rate",
            "investment_type",
            "acquisition_date",
            "percentage_of_class",
            "maturity",
            "amount_quantity",
            "cost",
            "fair_value",
        ]

    # print(master_df)
    if not master_df.empty:
        master_dedup_df = master_df.drop_duplicates(subset=dedup_cols, keep="first").reset_index(drop=True)
    else:
        master_dedup_df = master_df
    master_dedup_df.to_csv(OUTDIR / f"{filename}.csv", index=False)

main()

Normalize Cost:  3660
Normalize Cost:  2944
Normalize Cost:  6046
Normalize Cost:  1921
Normalize Cost:  52358
Normalize Cost:  1874
Normalize Cost:  4079
Normalize Cost:  <NA>
Normalize Cost:  <NA>
Normalize Cost:  1005
Normalize Cost:  1139
Normalize Cost:  1375
Normalize Cost:  10912
Normalize Cost:  2878
Normalize Cost:  4083
Normalize Cost:  2515
Normalize Cost:  2030
Normalize Cost:  255
Normalize Cost:  6555
Normalize Cost:  <NA>
Normalize Cost:  2463
Normalize Cost:  <NA>
Normalize Cost:  4150
Normalize Cost:  1988
Normalize Cost:  2257
Normalize Cost:  2175
Normalize Cost:  15812
Normalize Cost:  5195
Normalize Cost:  5815
Normalize Cost:  4637
Normalize Cost:  4000
Normalize Cost:  2945
Normalize Cost:  25325
Normalize Cost:  17000
Normalize Cost:  1991
Normalize Cost:  4264
Normalize Cost:  5546
Normalize Cost:  2437
Normalize Cost:  1999
Normalize Cost:  6985
Normalize Cost:  918
Normalize Cost:  1492
Normalize Cost:  905
Normalize Cost:  1557
Normalize Cost:  6349
Normaliz